# Contact Improvisation Movement Analysis — Exploratory Notebook

Exploration of interpersonal coordination dynamics in Contact Improvisation (CI) from video.
This notebook allows you to load videos, track dancers, compute kinematic and coordination measures, and compare with manual annotations.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.pose.pose_detector import PoseDetector
from src.tracking.dancer_tracker import TwoDancerTracker
from src.features.kinematics import extract_features_timeseries
from src.analysis.coordination import CrossCorrelationAnalysis, PhaseAnalysis, AnnotationManager
from src.visualization.plots import plot_trajectories, plot_movement_dynamics_dashboard, plot_phase_summary

print("CI Movement Analysis modules loaded successfully!")

## 1. Load Processed Features or Run on a Video
You can either load precomputed features or run the end-to-end pipeline.

In [ ]:
# Load precomputed features
features_path = Path("../outputs/ci_duet_sample_features.csv") if Path("../outputs").exists() else Path("outputs/ci_duet_sample_features.csv")
df_features = pd.read_csv(features_path)
print(f"Loaded {len(df_features)} frames of movement time series.")
df_features.head()

## 2. Load Manual Annotations
Compare manual intervals (e.g., contact, shared weight, transition) against computational features.

In [ ]:
ann_path = Path("../annotations/ci_duet_sample.csv") if Path("../annotations").exists() else Path("annotations/ci_duet_sample.csv")
ann_mgr = AnnotationManager(str(ann_path))
df_phases = ann_mgr.summarize_phases(df_features, fps=30.0)
df_phases

## 3. Visualize Movement Dynamics & Lagged Cross-Correlation
Inspect the synchronized time series: distances, speeds, directional similarity, and dynamic coupling.

In [ ]:
global_xcorr = CrossCorrelationAnalysis.compute_xcorr(
    df_features["torso_A_speed"].to_numpy(),
    df_features["torso_B_speed"].to_numpy(),
    fps=30.0,
    max_lag_sec=2.0
)
rolling_xcorr = CrossCorrelationAnalysis.compute_rolling_xcorr(
    df_features["torso_A_speed"].to_numpy(),
    df_features["torso_B_speed"].to_numpy(),
    fps=30.0,
    window_sec=4.0,
    step_sec=0.5,
    max_lag_sec=2.0
)

fig_dash = plot_movement_dynamics_dashboard(
    df_features,
    global_xcorr=global_xcorr,
    rolling_xcorr=rolling_xcorr,
    annotation_mgr=ann_mgr,
    title="CI Movement Dynamics Dashboard"
)
plt.show()

## 4. Phase-Wise Metric Differences
Directly test: **Do computational measures change systematically across manually identified phases?**

In [ ]:
fig_phase = plot_phase_summary(df_phases)
plt.show()